# Split Datasets

Split dataset to Train/Validation/Test sets

## A. Overview

Random split to Train, Validation and Test sets

In [2]:
%pip install datasketch sentence-transformers
# install either faiss-cuda or faiss-cpu depending on your environment
%pip install faiss-cuda
# %pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## B. Combine Datasets

In [1]:
from pathlib import Path
import csv
import os
import random
import json
import re

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import login

from datasketch import MinHash, MinHashLSH

import configuration
from src import data_utils, setup
from src.normalizer import similarity

RANDS_STATE = 42
random.seed(RANDS_STATE)

login(os.getenv("HF_TOKEN"))
dataset_path = Path('..') / 'data'

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### B.1. Load Datasets

In [4]:
# disaster_frac = data_utils.get_data_disaster_fraction()
# disaster_filepath = dataset_path / 'disaster' / str(disaster_frac)

# df_disaster_informative = pd.read_csv(disaster_filepath / 'informative.csv')
# df_disaster_informative['subset'] = 'disaster'
# # df_disaster_humanitarian = pd.read_csv(disaster_filepath / 'humanitarian.csv')
# # df_disaster_humanitarian['subset'] = 'disaster'

In [2]:
extended_filepath = dataset_path / "extended"

# df_weather = pd.read_csv(
#     extended_filepath / "weather.csv"
# )["tweet_text"].to_frame()
# df_weather["informative"] = False
# df_weather["subset"] = "weather"

df_out_topic = pd.read_csv(
    extended_filepath / "out_topic.csv"
)["tweet_text"].to_frame()
df_out_topic["informative"] = False
df_out_topic["subset"] = "out_topic"

### B.2. Combine Datasets

In [ ]:
# df_informative = (
#     pd.concat([df_disaster_informative, df_weather, df_out_topic], ignore_index=True)
#     .sample(frac=1, random_state=setup.RANDOM_SEED)
#     .reset_index(drop=True)
# )
# df_informative.head()

## C. Filtering

### C.1. Embedding

In [7]:
embedding_model="all-MiniLM-L6-v2"
nprobe = 10
chunk_size=50000
similarity_threshold=0.75
top_k = 20

In [ ]:
# # rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
# n_weather = len(df_weather)
# nlist_weather = int(4 * np.sqrt(n_weather))
# train_sample_size_weather = int(n_weather / 10)

# df_weather_embeddings = similarity.vectorize_faiss(
#     df_weather, text_column='tweet_text', model_name=embedding_model
# )

Loading embedding model 'all-MiniLM-L6-v2'...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorizing text to dense representations...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
# # save the embeddings to disk
# np.savez(
#     extended_filepath / "df_weather_embeddings.npz",
#     embeddings=df_weather_embeddings,
#     model_name=embedding_model,
# )

In [ ]:
# train_index_weather, gpu_index_weather = similarity.train_faiss_index(
#     df_weather_embeddings,
#     train_sample_size=train_sample_size_weather,
#     nprobe=nprobe,
# )

Building FAISS Index on CPU...
Transferring index to GPU for training...
Training index on 4751 samples...


WARNING clustering 4751 points to 871 centroids: please provide at least 33969 training points


In [ ]:
# df_weather_drop_indeices_radius = similarity.faiss_range_search(
#     df_weather_embeddings,
#     train_index_weather,
#     gpu_index=gpu_index_weather,
#     chunk_size=chunk_size,
#     similarity_threshold=similarity_threshold,
#     checkpoint_file='df_weather_drop_indeices_radius.checkpoint.json',
# )

Converting index back to CPU for range search...


Starting new FAISS radius search...
Processed up to row 47518/47518. Identified 18875 near-duplicates.
Remove checkpoint file upon successful completion.


In [ ]:
# df_weather.drop(index=df_weather_drop_indeices_radius).reset_index(drop=True).to_csv(
#     extended_filepath / f"df_weather_radius_{similarity_threshold}.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# df_weather_drop_indeices_knearest = similarity.faiss_neighbors_search(
#     df_weather_embeddings,
#     train_index_weather,
#     gpu_index=gpu_index_weather,
#     chunk_size=chunk_size,
#     top_k=top_k,
#     similarity_threshold=similarity_threshold,
#     checkpoint_file='df_weather_knearest.checkpoint.json',
# )

Starting new FAISS knearest search...


Processed up to row 47518/47518. Identified 18236 near-duplicates.
Remove checkpoint file upon successful completion.


In [ ]:
# df_weather.drop(index=df_weather_drop_indeices_knearest).reset_index(drop=True).to_csv(
#     extended_filepath / f"df_weather_knearest_{top_k}.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
n_out_topic = len(df_out_topic)
nlist_out_topic = int(4 * np.sqrt(n_out_topic))
train_sample_size_out_topic = int(n_out_topic / 10)

df_out_topic_embeddings = similarity.vectorize_faiss(
    df_out_topic, text_column='tweet_text', model_name=embedding_model
)

Loading embedding model 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
np.savez(
    extended_filepath / "df_out_topic_embeddings.npz",
    embeddings=df_out_topic_embeddings,
    model_name=embedding_model,
)

In [ ]:
train_index_out_topic, gpu_index_out_topic = similarity.train_faiss_index(
    df_out_topic_embeddings,
    train_sample_size=train_sample_size_out_topic,
    nprobe=nprobe,
)

In [ ]:
df_out_topic_drop_indeices_radius = similarity.faiss_range_search(
    df_out_topic_embeddings,
    train_index_out_topic,
    gpu_index=gpu_index_out_topic,
    chunk_size=chunk_size,
    similarity_threshold=similarity_threshold,
    checkpoint_file='df_out_topic_drop_indeices_radius.checkpoint.json',
)

In [ ]:
df_out_topic_knearest = similarity.faiss_neighbors_search(
    df_out_topic_embeddings,
    train_index_out_topic,
    gpu_index=gpu_index_out_topic,
    chunk_size=chunk_size,
    top_k=top_k,
    similarity_threshold=similarity_threshold,
    checkpoint_file='df_out_topic_knearest.checkpoint.json',
)